# Solvers: Euler and Dopri5

`CompiledModel.run` accepts `solver=` — `"euler"` (fixed step) or a diffrax
method (`"dopri5"`, `"tsit5"`, `"heun"`, or a solver instance). Adaptive
stepping reports `result.solver.num_steps`; dense output exposes
`result.evaluate(t)`. When an adaptive solve hits its step ceiling,
`result.solver.ok` is `False` instead of failing silently.

This infection rate is linear, so `S(t) = 999 exp(-0.3 t)` is the
picture to check the solvers against. Dopri5 should lie on that curve
and report far fewer steps than the fixed Euler grid. An off-grid save
time (t = 3.37) should still sit on the analytic curve, because the
solver interpolates. Dense output should retrace the saved susceptibles.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import numpy as np

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], 0.3))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())})


## Euler vs Dopri5 against the analytic S(t)

The reference Euler run uses 10,000 steps of `dt = 1e-4` and is only
saved at the endpoints — that is the accuracy check. The lines use a
coarser Euler so the shape is visible next to Dopri5 and the analytic
curve. The bars are the step counts for the reference comparison.


In [ ]:
# For this linear infection rate, S(t) = 999 * exp(-0.3 t).
ts = np.array([0.0, 1.0])
plan_ends = SavePlan(requests={"compartments": SaveRequest(Compartments(), ts=ts)})
euler = cm.run({}, y0, t0=0.0, steps=10_000, dt=1e-4, save=plan_ends, solver="euler")
dopri = cm.run(
    {}, y0, t0=0.0, t1=1.0, dt=0.1, save=plan_ends, solver="dopri5", rtol=1e-8, atol=1e-10
)
s_analytic = 999.0 * np.exp(-0.3)
np.testing.assert_allclose(float(np.asarray(dopri["compartments"].values.data)[-1, 0]), s_analytic, rtol=1e-5)
np.testing.assert_allclose(
    np.asarray(euler["compartments"].values.data)[-1],
    np.asarray(dopri["compartments"].values.data)[-1],
    rtol=1e-4,
    atol=1e-4,
)
assert dopri.solver is not None
assert int(dopri.solver.num_steps) != 10_000  # adaptive, not one step per Euler dt
assert int(dopri.solver.num_steps) < 10_000

grid = np.linspace(0.0, 1.0, 21)
plan_fig = SavePlan(requests={"compartments": SaveRequest(Compartments(), ts=grid)})
euler_fig = cm.run({}, y0, t0=0.0, steps=200, dt=0.005, save=plan_fig, solver="euler")
dopri_fig = cm.run(
    {}, y0, t0=0.0, t1=1.0, dt=0.1, save=plan_fig, solver="dopri5", rtol=1e-8, atol=1e-10
)
pd.DataFrame(
    {
        "analytic": 999.0 * np.exp(-0.3 * grid),
        "euler": np.asarray(euler_fig["compartments"].values.data)[:, 0],
        "dopri5": np.asarray(dopri_fig["compartments"].values.data)[:, 0],
    },
    index=grid,
).plot(
    title="S(t) = 999 exp(-0.3 t)",
    labels={"index": "time", "value": "susceptibles"},
)
pd.Series(
    {"euler (reference)": 10_000, "dopri5": int(dopri.solver.num_steps)}
).to_frame("steps").plot.bar(
    title="Dopri5 matches that S(1) in far fewer steps",
    labels={"index": "solver", "value": "steps"},
)


## Off-grid save times use the solver interpolant

The save times are 0, 3.37 and 10 — 3.37 is not a step boundary. The
markers of the saved series should sit on the analytic line.


In [ ]:
t_off = 3.37
plan_off = SavePlan(
    requests={"compartments": SaveRequest(Compartments(), ts=np.array([0.0, t_off, 10.0]))}
)
res = cm.run(
    {}, y0, t0=0.0, t1=10.0, dt=1.0, save=plan_off, solver="dopri5", rtol=1e-8, atol=1e-10
)
np.testing.assert_allclose(np.asarray(res["compartments"].times.values), [0.0, t_off, 10.0])
s_off = float(np.asarray(res["compartments"].values.data)[1, 0])
np.testing.assert_allclose(s_off, 999.0 * np.exp(-0.3 * t_off), rtol=1e-5)

grid = np.linspace(0.0, 10.0, 201)
saved_t = np.asarray(res["compartments"].times.values)
saved_s = np.asarray(res["compartments"].values.data)[:, 0]
shown = pd.DataFrame({"analytic": 999.0 * np.exp(-0.3 * grid)}, index=grid)
shown = shown.join(pd.Series(saved_s, index=saved_t, name="saved"), how="outer")
shown.plot(
    title="Off-grid save at t = 3.37 still lies on S(t)",
    labels={"index": "time", "value": "susceptibles"},
)


## Dense evaluation

`dense=True` keeps the interpolant after the solve. `evaluate(t)`
should retrace the susceptibles that were saved along the way.


In [ ]:
dense_plan = SavePlan(requests={"compartments": SaveRequest(Compartments())}, dense=True)
dense = cm.run(
    {}, y0, t0=0.0, t1=2.0, dt=0.1, save=dense_plan, solver="dopri5",
    rtol=1e-6, atol=1e-8, max_steps=512,
)
at0 = dense.evaluate(0.0)
np.testing.assert_allclose(np.asarray(at0.data), np.asarray(dense["compartments"].values.data)[0], rtol=1e-4)
assert dense.solver is not None and dense.solver.dense is True
print("euler steps (ref):", 10_000, "dopri5 steps:", int(dopri.solver.num_steps))

grid = np.linspace(0.0, 2.0, 41)
s_i = int(pmap.select_one(state["S"]))
evaluated = np.array(
    [float(np.asarray(dense.evaluate(float(t)).data)[s_i]) for t in grid]
)
saved = dense["compartments"].to_pandas()
s_col = [col for col in saved.columns if col.startswith("state=S")][0]
shown = pd.DataFrame({"evaluate(t)": evaluated}, index=grid)
shown = shown.join(saved[[s_col]].rename(columns={s_col: "saved S"}), how="outer")
shown.plot(
    title="Dense evaluate(t) retraces the saved susceptibles",
    labels={"index": "time", "value": "susceptibles"},
)


## Repeated Diffrax runs stay warm

Diffrax's `diffeqsolve` is `@eqx.filter_jit`. Summer4's vector field and save
callbacks are equinox Modules (structural equality on the model digest and
save requests), so a second `run` with the same plan and solver reuses the
compiled solve instead of retracing.

Equinox also treats plain Python `float` leaves as static. `prepare()` promotes
those leaves to float64 arrays, so a host loop that passes `{str: float}`
dicts (as calibration draws do) still hits the cache. The claim below is that
warm median time stays under 50 ms across **distinct** float `beta` values.


In [ ]:
import statistics
import time

import jax
import diffrax

from summer4 import Param

state_p = Property("state", ("S", "I", "R"))
pmap_p = PropertyMap.from_property(state_p)
model_p = FlowModel(pmap_p)
model_p.add_flow(TransitionFlow("infection", state_p["S"], state_p["I"], Param("beta")))
model_p.add_flow(TransitionFlow("recovery", state_p["I"], state_p["R"], 0.1))
cm_p = model_p.compile()
y0_p = PropertyData.wrap(pmap_p, np.array([999.0, 1.0, 0.0]))

ts_warm = np.linspace(0.0, 20.0, 201)
plan_warm = SavePlan(requests={"compartments": SaveRequest(Compartments(), ts=ts_warm)})
betas = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]


def _warm_once(beta: float) -> None:
    warm = cm_p.run(
        {"beta": beta},
        y0_p,
        t0=0.0,
        t1=20.0,
        dt=0.1,
        save=plan_warm,
        solver=diffrax.Euler(),
        max_steps=4096,
    )
    jax.block_until_ready(warm["compartments"].values.data)


_warm_once(betas[0])  # discard compile
times = []
for beta in betas[1:]:
    t0 = time.perf_counter()
    _warm_once(beta)
    times.append(time.perf_counter() - t0)

# Pre-fix path retraced every call (~0.3–1.8 s). Warm median stays under 50 ms.
assert statistics.median(times) < 0.05


## Solver failure is visible

Adaptive solves default to `throw=False`, so a step ceiling that is too low
does not raise — it returns a trajectory and sets `result.solver.ok` to
`False`. That flag is a traced boolean (`result_code == 0`), so a jitted
calibration loss can return `-inf` when the solve failed.

Diffrax pre-fills unsaved times with `inf`. A tiny ceiling that only rejects
steps leaves the whole series as `inf`; a slightly larger one (here
`max_steps=16`) accepts some steps, fills early save times with finite
values, then leaves the rest as `inf` once the budget is gone. The figure
shows that partial progress; the bars compare step counts to a healthy run.


In [ ]:
tight = cm.run(
    {},
    y0,
    t0=0.0,
    t1=20.0,
    dt=0.1,
    save=plan,
    solver="dopri5",
    rtol=1e-8,
    atol=1e-10,
    max_steps=16,
)
healthy = cm.run(
    {},
    y0,
    t0=0.0,
    t1=20.0,
    dt=0.1,
    save=plan,
    solver="dopri5",
    rtol=1e-8,
    atol=1e-10,
)
assert tight.solver is not None and healthy.solver is not None
assert bool(healthy.solver.ok) is True
assert bool(tight.solver.ok) is False
assert int(tight.solver.result_code) != 0
assert int(tight.solver.num_accepted_steps) > 0

s_tight = np.asarray(tight["compartments"].values.data)[:, 0]
s_healthy = np.asarray(healthy["compartments"].values.data)[:, 0]
times = np.asarray(tight["compartments"].times.values)
finite = np.isfinite(s_tight)
assert finite.any() and (~finite).any()
assert finite[0] and not finite[-1]

pd.DataFrame(
    {
        "time": np.concatenate([times[finite], times]),
        "S": np.concatenate([s_tight[finite], s_healthy]),
        "run": (["max_steps=16 (finite saves)"] * int(finite.sum()))
        + (["healthy"] * len(times)),
    }
).plot(
    x="time",
    y="S",
    color="run",
    title="Partial trajectory before the step ceiling",
    labels={"time": "t", "S": "susceptibles"},
)
